In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Inicio rápido con Estimator

El primitivo Estimator calcula los valores de expectación para uno o más observables con respecto a los estados preparados por circuits cuánticos. Los circuits pueden estar parametrizados, siempre que los valores de los parámetros también se proporcionen como entrada al primitivo.

Este primitivo tiene varias [técnicas de mitigación y supresión de errores](/guides/error-mitigation-and-suppression-techniques) integradas, incluyendo desacoplamiento dinámico, Pauli-twirling, ZNE por plegado de gates, PEA y PEC. También admite una opción `resilience_level` que te permite configurar fácilmente el equilibrio entre costo y precisión.
Los pasos de este tema describen cómo configurar Estimator, explorar las opciones que puedes usar para configurarlo e invocarlo en un programa.


<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o más recientes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

Esta es una función experimental y podría cambiar en el futuro.

:::

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

print(backend.name)

ibm_fez


### 2. Create a circuit and an observable

You need at least one circuit and one observable as inputs to the Estimator primitive.

In [2]:
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp

entanglement = [tuple(edge) for edge in backend.coupling_map.get_edges()]
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 0.5) for i, j in entanglement],
    num_qubits=backend.num_qubits,
)
circuit = qaoa_ansatz(observable, reps=2)
# The circuit is parametrized, so we will define the parameter values for execution
param_values = [0.1, 0.2, 0.3, 0.4]

### 2. Crear un circuit y un observable
Necesitas al menos un circuit y un observable como entradas para el primitivo Estimator.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(f">>> Circuit ops (ISA): {isa_circuit.count_ops()}")

>>> Circuit ops (ISA): OrderedDict([('rz', 4472), ('sx', 1884), ('cz', 1120)])


El circuit y el observable deben transformarse para usar solo instrucciones admitidas por la QPU (denominadas circuits de *arquitectura de conjunto de instrucciones (ISA)*). Usa el transpilador para hacer esto.

In [4]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)

### 4. Invoke Estimator and get results

Next, invoke the `run()` method to calculate expectation values for the input circuits and observables. The circuit, observable, and optional parameter value sets are input as *primitive unified bloc* (PUB) tuples.

In [5]:
job = estimator.run([(isa_circuit, isa_observable, param_values)])
print(f">>> Job ID: {job.job_id()}")
print(f">>> Job Status: {job.status()}")

>>> Job ID: d82869ntjchs73bnokog


>>> Job Status: QUEUED


In [6]:
result = job.result()
print(f">>> {result}")
print(f"  > Expectation value: {result[0].data.evs}")
print(f"  > Metadata: {result[0].metadata}")

>>> PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})
  > Expectation value: 30.60337496305257
  > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


### 4. Invocar Estimator y obtener resultados
A continuación, invoca el método `run()` para calcular los valores de expectación para los circuits y observables de entrada. El circuit, el observable y los conjuntos opcionales de valores de parámetros se ingresan como tuplas de *bloque unificado primitivo* (PUB).